# Task 2 & 3: Data Ingestion, HDFS Upload and Spark Processing
## RSNA Bone Age Dataset — Big Data Pipeline

## Step 1: Initialize PySpark Session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count, stddev, min, max, when, isnan
from pyspark.sql.types import *
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Initialize Spark Session
spark = SparkSession.builder \
    .appName('RSNA_BoneAge_Pipeline') \
    .master('local[*]') \
    .config('spark.sql.warehouse.dir', '/user/hive/warehouse') \
    .config('spark.driver.memory', '4g') \
    .config('spark.executor.memory', '4g') \
    .enableHiveSupport() \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')
print(f'Spark version: {spark.version}')
print('SparkSession initialized successfully.')

## Step 2: Load CSV Metadata from HDFS

In [ ]:
# Define schema for RSNA Bone Age metadata
schema = StructType([
    StructField('id',       IntegerType(), True),
    StructField('boneage', FloatType(),   True),
    StructField('male',    BooleanType(), True)
])

# Load training metadata CSV
df_train = spark.read.csv(
    '/data/boneage-training-dataset.csv',
    header=True,
    schema=schema
)

print(f'Total training records: {df_train.count()}')
df_train.printSchema()
df_train.show(10)

## Step 3: Data Wrangling and Transformations

In [ ]:
# Check for null values
print('=== Null Value Check ===')
df_train.select([count(when(col(c).isNull(), c)).alias(c) for c in df_train.columns]).show()

# Add derived columns
df_enriched = df_train \
    .withColumn('boneage_years', col('boneage') / 12.0) \
    .withColumn('gender', when(col('male') == True, 'Male').otherwise('Female')) \
    .withColumn('age_group',
        when(col('boneage') < 60,  'Infant (0-5y)')
        .when(col('boneage') < 120, 'Child (5-10y)')
        .when(col('boneage') < 168, 'Pre-teen (10-14y)')
        .otherwise('Teenager (14y+)'))

print('=== Enriched Dataset Sample ===')
df_enriched.show(10)
print(f'Total enriched records: {df_enriched.count()}')

## Step 4: Exploratory Statistics with Spark

In [ ]:
# Summary statistics
print('=== Summary Statistics ===')
df_enriched.describe(['boneage', 'boneage_years']).show()

# Gender distribution
print('=== Gender Distribution ===')
df_enriched.groupBy('gender').agg(
    count('id').alias('count'),
    avg('boneage').alias('avg_boneage_months'),
    stddev('boneage').alias('stddev_boneage')
).show()

# Age group distribution
print('=== Age Group Distribution ===')
df_enriched.groupBy('age_group').agg(
    count('id').alias('count'),
    avg('boneage').alias('avg_boneage')
).orderBy('avg_boneage').show()

# Min/Max per gender
print('=== Min/Max Bone Age by Gender ===')
df_enriched.groupBy('gender').agg(
    min('boneage').alias('min_months'),
    max('boneage').alias('max_months')
).show()

## Step 5: Write Processed Data to Hive Table

In [ ]:
# Create Hive database
spark.sql('CREATE DATABASE IF NOT EXISTS boneage_db')
spark.sql('USE boneage_db')

# Write enriched dataframe as Hive table
df_enriched.write \
    .mode('overwrite') \
    .saveAsTable('boneage_db.patient_records')

print('Hive table created: boneage_db.patient_records')
spark.sql('SHOW TABLES IN boneage_db').show()
spark.sql('SELECT COUNT(*) as total FROM boneage_db.patient_records').show()

## Step 6: HiveQL Queries for Exploratory Analysis

In [ ]:
# HiveQL Query 1: Average bone age by gender
print('=== HiveQL Query 1: Average Bone Age by Gender ===')
spark.sql('''
    SELECT gender,
           COUNT(*) AS patient_count,
           ROUND(AVG(boneage), 2) AS avg_bone_age_months,
           ROUND(AVG(boneage_years), 2) AS avg_bone_age_years
    FROM boneage_db.patient_records
    GROUP BY gender
    ORDER BY avg_bone_age_months DESC
''').show()

# HiveQL Query 2: Patient count by age group
print('=== HiveQL Query 2: Patient Distribution by Age Group ===')
spark.sql('''
    SELECT age_group,
           COUNT(*) AS patient_count,
           ROUND(MIN(boneage), 1) AS min_months,
           ROUND(MAX(boneage), 1) AS max_months
    FROM boneage_db.patient_records
    GROUP BY age_group
    ORDER BY min_months
''').show()

# HiveQL Query 3: Gender breakdown within age groups
print('=== HiveQL Query 3: Gender Breakdown Within Age Groups ===')
spark.sql('''
    SELECT age_group, gender,
           COUNT(*) AS count,
           ROUND(AVG(boneage), 2) AS avg_boneage
    FROM boneage_db.patient_records
    GROUP BY age_group, gender
    ORDER BY age_group, gender
''').show(20)

## Step 7: Spark vs Hive Performance Comparison

In [ ]:
import time

# Spark DataFrame API timing
start = time.time()
result_spark = df_enriched.groupBy('gender').agg(
    count('id').alias('count'),
    avg('boneage').alias('avg_boneage')
).collect()
spark_time = time.time() - start

# HiveQL timing
start = time.time()
result_hive = spark.sql('''
    SELECT gender, COUNT(*) as count, AVG(boneage) as avg_boneage
    FROM boneage_db.patient_records
    GROUP BY gender
''').collect()
hive_time = time.time() - start

print(f'Spark DataFrame API execution time: {spark_time:.3f} seconds')
print(f'HiveQL execution time:              {hive_time:.3f} seconds')
print()
print('=== Comparison Summary ===')
print('Spark DataFrame API: Better for iterative ML pipelines and real-time processing.')
print('HiveQL: Better for batch analytics, SQL-familiar users, and structured reporting.')

## Step 8: Visualisations

In [ ]:
# Convert to Pandas for visualisation
pdf = df_enriched.toPandas()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('RSNA Bone Age Dataset — Exploratory Analysis', fontsize=16)

# Plot 1: Bone age distribution
axes[0,0].hist(pdf['boneage'], bins=30, color='steelblue', edgecolor='white')
axes[0,0].set_title('Bone Age Distribution (months)')
axes[0,0].set_xlabel('Bone Age (months)')
axes[0,0].set_ylabel('Frequency')

# Plot 2: Gender distribution
gender_counts = pdf['gender'].value_counts()
axes[0,1].bar(gender_counts.index, gender_counts.values, color=['steelblue','salmon'])
axes[0,1].set_title('Gender Distribution')
axes[0,1].set_ylabel('Count')

# Plot 3: Bone age by gender (boxplot)
pdf.boxplot(column='boneage', by='gender', ax=axes[1,0])
axes[1,0].set_title('Bone Age by Gender')
axes[1,0].set_xlabel('Gender')
axes[1,0].set_ylabel('Bone Age (months)')

# Plot 4: Age group distribution
age_counts = pdf['age_group'].value_counts()
axes[1,1].bar(age_counts.index, age_counts.values, color='teal')
axes[1,1].set_title('Patient Count by Age Group')
axes[1,1].set_ylabel('Count')
axes[1,1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('/home/jovyan/work/eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('EDA plots saved.')